# RESUME LoRA fine-tune `qwen2.5:3b-instruct` v6 — продолжение с checkpoint-1425

**Когда использовать:** предыдущий запуск дошёл до checkpoint-1425 (3 эпохи) и упёрся в Kaggle 12-часовой лимит. У тебя сохранены: `adapter_config.json`, `adapter_model.safetensors`, `optimizer.pt`, `scaler.pt`, `trainer_state.json`, `training_args.bin`, `tokenizer*`.

**Что делает этот notebook:**
1. Загружает базовую Qwen 2.5 (3B) + применяет твою LoRA-конфигурацию (R=24, α=48)
2. Вызывает `trainer.train(resume_from_checkpoint=...)` — HF Trainer автоматически:
   - Восстановит веса LoRA из `adapter_model.safetensors`
   - Восстановит состояние Adam (momentum/variance) из `optimizer.pt`
   - Восстановит scaler из `scaler.pt`
   - Прочитает `trainer_state.json` → продолжит с шага 1425
   - LR scheduler перестроится из текущих args (идентичных оригинальным) → продолжит ту же cosine-кривую
3. Дотренирует оставшиеся 950 шагов (с 1425 до 2375 = эпохи 4 и 5) — ~7.5 ч на T4
4. Если Kaggle убьёт сессию на ~12 ч → у тебя будет `checkpoint-1900` (конец эпохи 4)
5. Сохранит финальный LoRA + соберёт GGUF Q8_0

**Kaggle setup:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add Data → 3 Kaggle Datasets:
   - `train.jsonl` + `eval.jsonl`
   - LoRA checkpoint-1425 (5+ файлов: adapter_config, adapter_model.safetensors, optimizer.pt, scaler.pt, trainer_state.json, training_args.bin, tokenizer*)
4. **Save Version → Save & Run All (Commit)** — фоновый запуск.


## 1. Install

In [ ]:
!pip install -q --upgrade unsloth
print('install done')

## 2. Detect environment + locate train.jsonl + eval.jsonl + LoRA checkpoint

Auto-detect Kaggle vs Colab. Ищет LoRA checkpoint по `adapter_config.json` в `/kaggle/input/`.

In [ ]:
import os, glob, shutil

if os.path.isdir('/kaggle/working'):
    ENV = 'kaggle'
    WORK_DIR = '/kaggle/working'
    train_candidates = glob.glob('/kaggle/input/**/train.jsonl', recursive=True)
    eval_candidates = glob.glob('/kaggle/input/**/eval.jsonl', recursive=True)
    assert train_candidates, '❌ train.jsonl не найден в /kaggle/input/. Add Data → загрузи Dataset с jsonl.'
    assert eval_candidates, '❌ eval.jsonl не найден.'
    shutil.copy(train_candidates[0], f'{WORK_DIR}/train.jsonl')
    shutil.copy(eval_candidates[0], f'{WORK_DIR}/eval.jsonl')
elif os.path.isdir('/content'):
    ENV = 'colab'
    WORK_DIR = '/content'
    assert os.path.isfile(f'{WORK_DIR}/train.jsonl'), '❌ /content/train.jsonl не найден.'
    assert os.path.isfile(f'{WORK_DIR}/eval.jsonl'), '❌ /content/eval.jsonl не найден.'
else:
    raise RuntimeError('Среда не определена. Только Kaggle / Colab.')

# Найти папку с LoRA checkpoint (по adapter_config.json)
if ENV == 'kaggle':
    cfg_candidates = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
else:
    cfg_candidates = glob.glob('/content/**/adapter_config.json', recursive=True)
assert cfg_candidates, '❌ adapter_config.json не найден. Загрузи LoRA checkpoint как Dataset.'
RESUME_CHECKPOINT_DIR = os.path.dirname(cfg_candidates[0])
print(f'Environment: {ENV}')
print(f'Work dir: {WORK_DIR}')
print(f'Resume checkpoint dir: {RESUME_CHECKPOINT_DIR}')

# Sanity check — список файлов в checkpoint
print('\nФайлы в checkpoint:')
for f in sorted(os.listdir(RESUME_CHECKPOINT_DIR)):
    sz = os.path.getsize(os.path.join(RESUME_CHECKPOINT_DIR, f))
    print(f'  {f}: {sz/1e6:.1f} MB')

os.chdir(WORK_DIR)
print(f'\ntrain.jsonl: {sum(1 for _ in open(f"{WORK_DIR}/train.jsonl"))} lines')
print(f'eval.jsonl:  {sum(1 for _ in open(f"{WORK_DIR}/eval.jsonl"))} lines')

## 3. Config — должен совпадать с оригинальным запуском

**Важно:** все гиперпараметры точно как при оригинальном обучении. Иначе LR scheduler пересчитается некорректно.

In [ ]:
BASE_MODEL = 'unsloth/Qwen2.5-3B-Instruct'
MAX_SEQ_LEN = 8192

LORA_R = 24
LORA_ALPHA = 48
LORA_DROPOUT = 0.0

BATCH = 2
GRAD_ACCUM = 4
EPOCHS = 5          # ТО ЖЕ, что в оригинале — Trainer перестроит scheduler идентично
LR = 1.5e-4
WARMUP = 30
WEIGHT_DECAY = 0.01
LOG_EVERY = 20

OUTPUT_DIR = f'{WORK_DIR}/outputs'
GGUF_NAME = 'qwen2.5-3b-furniture-v6'
GGUF_QUANT = 'q8_0'

## 4. Load base model + apply LoRA config (веса будут восстановлены Trainer'ом)

Здесь только применяем LoRA-конфигурацию. Сами веса LoRA восстановятся на шаге `resume_from_checkpoint`.

In [ ]:
import pandas  # noqa: F401
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 5. Prepare dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset('json', data_files=f'{WORK_DIR}/train.jsonl', split='train')
print(f'train examples: {len(raw)}')

def to_text(example):
    return {'text': tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False)}

train_ds = raw.map(to_text, remove_columns=raw.column_names)
print('rendered first 200 chars:')
print(train_ds[0]['text'][:200])

## 6. RESUME train — продолжаем с checkpoint-1425

Trainer прочитает `trainer_state.json` → global_step=1425, восстановит optimizer/scaler, LR scheduler перестроит с теми же args (идентично оригинальной cosine-кривой). Тренировка пойдёт с шага 1426 до шага 2375 (~950 шагов ≈ 7.5 ч на T4).

Если Kaggle убьёт сессию около 12 часов — `save_strategy='epoch'` означает, что `checkpoint-1900` (конец эпохи 4) уже будет сохранён в `/kaggle/working/outputs/`.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        warmup_steps=WARMUP,
        weight_decay=WEIGHT_DECAY,
        logging_steps=LOG_EVERY,
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        seed=42,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        output_dir=OUTPUT_DIR,
        report_to='none',
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        save_strategy='epoch',
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

# КЛЮЧЕВОЕ: resume_from_checkpoint=<dir> загружает adapter, optimizer, scaler, trainer_state
stats = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT_DIR)
print(f'\nTrain done. Loss: {stats.training_loss:.4f}, time: {stats.metrics["train_runtime"]/60:.1f} min')

## 7. Eval — exact match + tool-sequence accuracy

Опционально. Если хватает времени — прогоняем на 198 eval-примерах.

In [ ]:
import json, re
from collections import Counter

FastLanguageModel.for_inference(model)

with open(f'{WORK_DIR}/eval.jsonl') as f:
    eval_lines = [json.loads(l) for l in f]

def extract_plan(text):
    text = text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```\w*\n?', '', text).rstrip('`').strip()
    try:
        return json.loads(text).get('plan')
    except Exception:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if not m:
            return None
        try:
            return json.loads(m.group(0)).get('plan')
        except Exception:
            return None

exact = 0
tool_match = 0
for ex in eval_lines:
    msgs = ex['messages'][:-1]
    gold = ex['messages'][-1]['content']
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
    pred = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    if pred.strip() == gold.strip():
        exact += 1
    gold_plan = extract_plan(gold)
    pred_plan = extract_plan(pred)
    if gold_plan and pred_plan:
        if [s.get('tool') for s in gold_plan] == [s.get('tool') for s in pred_plan]:
            tool_match += 1

N = len(eval_lines)
print(f'Exact match: {exact}/{N} = {100*exact/N:.1f}%')
print(f'Tool sequence match: {tool_match}/{N} = {100*tool_match/N:.1f}%')

## 8. Build llama.cpp (~5 min)

In [ ]:
!apt-get install -y -qq build-essential cmake libcurl4-openssl-dev 2>&1 | tail -3
!rm -rf /root/.unsloth/llama.cpp
!mkdir -p /root/.unsloth
!git clone --depth=1 https://github.com/ggerganov/llama.cpp /root/.unsloth/llama.cpp 2>&1 | tail -3
!cd /root/.unsloth/llama.cpp && cmake -B build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF 2>&1 | tail -5
!cd /root/.unsloth/llama.cpp && cmake --build build --config Release -j 2 --target llama-quantize 2>&1 | tail -10
!ls -la /root/.unsloth/llama.cpp/build/bin/llama-quantize

## 9. Save LoRA adapter + merge → GGUF f16 → Q8_0

In [ ]:
LORA_DIR = f'{WORK_DIR}/{GGUF_NAME}-lora'
MERGED_DIR = f'{WORK_DIR}/{GGUF_NAME}-merged'
F16_GGUF = f'{WORK_DIR}/{GGUF_NAME}-f16.gguf'
FINAL_GGUF = f'{WORK_DIR}/{GGUF_NAME}-{GGUF_QUANT}.gguf'

# 0. Save LoRA adapter (для warm-start будущих iterations)
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f'\n✓ LoRA adapter saved to {LORA_DIR}')
!ls -lh {LORA_DIR}

# 1. Merge LoRA в base → HF dir
print('\nMerging LoRA into base model (16-bit)...')
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print('✓ merged 16-bit saved')

# 2. HF → GGUF f16
print('\nConverting HF → GGUF f16 (~5 min)...')
!pip install -q sentencepiece protobuf
!python /root/.unsloth/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {F16_GGUF} --outtype f16 2>&1 | tail -5
!ls -lh {F16_GGUF}

# 3. f16 → Q8_0
print('\nQuantizing f16 → Q8_0 (~2 min)...')
!/root/.unsloth/llama.cpp/build/bin/llama-quantize {F16_GGUF} {FINAL_GGUF} {GGUF_QUANT} 2>&1 | tail -10
!ls -lh {FINAL_GGUF}

## 10. Modelfile + cleanup

In [ ]:
import os, shutil

modelfile = f'''FROM ./{GGUF_NAME}-{GGUF_QUANT}.gguf

TEMPLATE """{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{ if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
{{{{ .Response }}}}<|im_end|>
"""

PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
PARAMETER temperature 0.0
PARAMETER num_ctx 8192
'''
with open(f'{WORK_DIR}/Modelfile', 'w') as f:
    f.write(modelfile)
print('✓ Modelfile written')

TRASH = (MERGED_DIR, F16_GGUF, OUTPUT_DIR)
for path in TRASH:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f'  removed dir: {path}')
    elif os.path.isfile(path):
        os.remove(path)
        print(f'  removed file: {path}')

print(f'\n=== ИТОГ {WORK_DIR}/ ===')
!ls -lh {WORK_DIR}/
print('\n=== Размеры ===')
!du -sh {WORK_DIR}/*

## 11. Что делать после Run All

Если Kaggle убил сессию ДО конца Cell #9 (обучение не успело завершиться):
- В `/kaggle/working/outputs/` будет `checkpoint-1900` (конец эпохи 4)
- Используй его так же, как сейчас используешь checkpoint-1425: либо сразу запусти merge-notebook (`notebook35d9a7a07c.ipynb`), либо снова resume через этот notebook ещё на 1 эпоху

Если обучение завершилось — в `/kaggle/working/` будут:
- `qwen2.5-3b-furniture-v6-q8_0.gguf` (~3.3GB) — основное
- `Modelfile` — конфиг
- `qwen2.5-3b-furniture-v6-lora/` — для warm-start

**Локально:**
```bash
cd ~/Downloads
ollama create furniture-3b-v6 -f Modelfile
ollama run furniture-3b-v6 "найди диваны до 50000"
```

После проверки — поменять `docker-compose.yml`:
```yaml
OLLAMA_MODEL: furniture-3b-v6
```
```bash
docker compose restart backend
```